### Dialogue to deeal with the admin routes for my blog site.  This includes editing and saving blog posts, though not as yet creating from scratch

Features still to be added
1. buttons on the relevant blog pages to allow edit - Done
2. buttons on the relevant blog pages to allow delete - Done
3. buttons on the relevant blog pages to allow download - Done
4. Add the capability to change tags for a post _still to do


In [ ]:
#| default_exp admin_v1

In [ ]:
#| hide
from nbdev.showdoc import *
import nbdev

In [ ]:
#| export
from fastlite import Database
from pathlib import Path
from datetime import datetime, timedelta
import my_blog.config as config
from urllib.parse import quote, unquote
from fasthtml.common import *
from monsterui.all import *
from fasthtml_auth import AuthManager
from fasthtml.jupyter import *
import re
import frontmatter

In [ ]:
#| export
from my_blog.core_v5 import (layout, render_md, EnhancedRenderer, process_obsidian_images, preprocess_markdown, process_strava_embeddings, 
    process_komoot_embed, process_bunny_embed, navbar, footer, hx_link, hx_attrs, process_cdn_images)

In [ ]:
#| export
# Route collection for deferred registration
_admin_routes = []

def route(path=None):
    """Decorator to collect routes without registering them immediately.
    Use @route('/path') or @route() for function-name-based paths."""
    def decorator(f):
        _admin_routes.append((path, f))
        return f
    if callable(path):  # @route without parens
        f, path = path, None
        _admin_routes.append((path, f))
        return f
    return decorator

In [ ]:
#| export
_state = None

In [ ]:
#| export
def register_admin_routes(app, state):
    """Register all collected routes with the app and setting state for the module"""
    global _state
    _state = state
    for path, handler in _admin_routes:
        if path:
            app.route(path)(handler)
        else:
            app.route(handler)

In [ ]:
#| export
def require_admin(req):
    user = req.scope.get('user')
    if not user or user.role != 'admin':
        return Response("Forbidden", status_code=403)


In [ ]:
#| export
@route('/admin/edit/{slug}')
def post_edit(req, htmx, slug: str):
    # Load post content from database
    if err := require_admin(req): return err
    p = load_post(slug)
    if not post:
        return layout(req, H2("Not Found"), P("Post not found."), title="Not Found", htmx=htmx)
    content = p['content']

    # Create edit page components and layout including textarea, output div, buttons for cancel, full_preview, save, save and return
    # main content area has id: 'main-content'
    frm = Form(
        Div(id='notify'),
        Grid(cls='grid grid-cols-2 gap-4')(
        Textarea(content, name='raw_post', id='raw_post', hx_post= f'/admin/post_simple_render', hx_target='#preview', hx_trigger='load, keyup changed delay:500ms',
        cls='w-full min-h-[80vh] text-lg overflow-hidden'),
        Div(id='preview', cls='text-lg border rounded p-4')
        ),
        Div(
            # Button('Cancel', cls=[ButtonT.secondary, ButtonT.sm], href=f'/blog/{slug}'),
            A('Cancel', href=f'/blog/{slug}', cls=[ButtonT.secondary, ButtonT.sm]),
            Button('Save', hx_post=f'/admin/save/{slug}', hx_target='#notify', hx_swap='innerHTML', cls=[ButtonT.primary, ButtonT.sm]),
            Button('Full Preview', cls=[ButtonT.primary, ButtonT.sm], hx_post=f'/admin/post_simple_render?full_render=True', 
            hx_target='#preview'),
            Button('Save & return', hx_post=f'/admin/save/{slug}?return_to_blog=true', hx_target='#notify', hx_swap='innerHTML', cls=[ButtonT.primary, ButtonT.sm]),           
            id='button_bar'
        ),
        Hidden(name='slug', value=slug)
    )
    return edit_layout(req, H1(p['title'], cls="text-3xl font-bold mb-2"), Span(p['created'].strftime('%B %d, %Y'), cls="text-muted-foreground text-sm mb-8 block"), 
        frm, title=p['title'], htmx=htmx)

    # 

In [ ]:
#| export
@route('/admin/save/{slug}')
def save_post(htmx, slug: str, raw_post: str, return_to_blog: bool = False):
    p = load_post(slug)
    post_id = p['id']
    content = raw_post
    try: 
        _state.posts_t.update(dict(id=post_id, content=content, updated=datetime.now()))
    except Exception as inst:
        return Toast(f"Unable to save post: {inst}", alert_cls=AlertT.error)
    if return_to_blog:
        return HtmxResponseHeaders(redirect=f'/blog/{slug}')
    else:
        return Toast("Post saved :)", alert_cls=AlertT.info)


In [ ]:
#| export
@route('/admin/delete/{slug}')
def delete_post(req,htmx, slug: str):
    if err := require_admin(req): return err
    p = load_post(slug)
    if not p: return Toast("Post not found", alert_cls=AlertT.error)
    try:
        operation = "deleting post tags"
        _state.pdb.execute("DELETE FROM post_tags WHERE post_id = ?", [p['id']])
        operation = "deleting post"
        _state.posts_t.delete(p['id'])
    except Exception as e:
        return Toast(f"Error {e} with operation: {operation}", alert_cls=AlertT.error)
    return HtmxResponseHeaders(redirect='/blog')

In [ ]:
#| export
@route('/admin/download/{slug}')
def download_post(req, slug: str):
    if err := require_admin(req): return err
    p = load_post(slug)
    if not p: return Response("Post not found", status_code=404)
    tag_rows = _state.pdb.q("""SELECT t.name FROM tags t JOIN post_tags pt ON t.id = pt.tag_id WHERE pt.post_id = ?""", [p['id']])
    tags = [r['name'] for r in tag_rows]
    fm = dict(title=p['title'], tags=tags, excerpt=p.get('excerpt', ''), created=str(p['created']))
    fm['private'] = bool(p.get('private', False))
    if p.get('updated'): fm['updated'] = str(p['updated'])
    post = frontmatter.Post(p['content'], **fm)
    md = frontmatter.dumps(post)
    fname = f"{p['title']}.md"
    return Response(md, media_type='text/markdown', headers={'Content-Disposition': f'attachment; filename="{fname}"'})

In [ ]:
#| export
def replace_iframe(content: str):
    """ Comments re regex: The [^>] means not >, and so adding the * means everything up to the >.  The ? .*?< makes the .* non-greedy (lazy). Without it, if there were two iframes in the content, .* would greedily match from the first <iframe> all the way to the last </iframe>, swallowing everything in between — including any content between the two iframes. With .*?, it stops at the nearest </iframe>."""

    pattern = r'<iframe[^>]*>.*?</iframe>'

    content = re.sub(pattern, '<div class="border rounded p-4 text-center text-muted-foreground">iframe embed (full preview to see)</div>', content, flags=re.DOTALL)
    return content

In [ ]:
#| export
def replace_strava(content: str):
    """ The strava pattern is: {{strava:16611889793}}
    """
    pattern = r'\{\{strava:\d+\}\}'

    content = re.sub(pattern, '<div class="border rounded p-4 text-center text-muted-foreground">🚴 Strava embed (full preview to see)</div>', content, flags=re.DOTALL)
    return content

In [ ]:
test_strava = 'This is dummy text with a strava embed {{strava:1234567}} here'
expected = 'This is dummy text with a strava embed <div class="border rounded p-4 text-center text-muted-foreground">🚴 Strava embed (full preview to see)</div> here'
assert replace_strava(test_strava)==expected

In [ ]:
test_iframe_content = 'This is dummy text with an iframe <iframe src="https://www.google.com/maps/d/embed?mid=1NXlWhw7ZUcEQ_hGaXVAuuPz4DnGN_wQ&hl=en&ehbc=2E312F" width="640" height="480"></iframe> embedded'
expected = 'This is dummy text with an iframe <div class="border rounded p-4 text-center text-muted-foreground">iframe embed (full preview to see)</div> embedded'
assert replace_iframe(test_iframe_content) == expected

In [ ]:
#| export
@route('/admin/post_simple_render')
def post(raw_post: str, slug: str, full_render: bool = False):
    image_base = f"/static/image/post_images/{slug}"
    if full_render:
        content = preprocess_markdown(raw_post, image_base=image_base)
        content = render_md(content, renderer=EnhancedRenderer)
        content = process_cdn_images(content)
        content = process_strava_embeddings(content)
        content = process_komoot_embed(content)
        content = process_bunny_embed(content, slug)
        return Div(content, Script(src="https://strava-embeds.com/embed.js"))
    else:
        content = replace_iframe(raw_post)
        content = replace_strava(content)
        content = preprocess_markdown(content, image_base=image_base)
        content = render_md(content, renderer=EnhancedRenderer)
        content = process_cdn_images(content)
        content = NotStr(replace_iframe(str(content)))   # catch iframes generated by komoot etc post-render
        return Div(content)


In [ ]:
#| export
def load_post(slug):
    row = _state.posts_t.rows_where("slug = ?", [slug], limit=1)
    p = next((dict(r) for r in row), None)
    if not p: return None
    p['created'] = datetime.fromisoformat(p['created']) if isinstance(p['created'], str) else p['created']
    return p

In [ ]:
#| export
def edit_layout(req, *content, htmx, title=None):
    if htmx and htmx.request: return (Title(title), *content)
    main = Main(*content, cls='w-full max-w-full px-8 mx-auto py-8 space-y-8', id="main-content")
    return Title(title), Div(Div(navbar(req), cls='max-w-full px-8 mx-auto mt-4'), main, footer(), cls="flex flex-col min-h-screen")

### Export

In [ ]:
nbdev.nbdev_export()